# 04_子 Agent

**来源:** [LangChain Docs — Subagents](https://docs.langchain.com/oss/python/deepagents/subagents)

本 Notebook 演示 Deep Agents 的子 Agent 功能。子 Agent 用于委托工作，保持主 Agent 上下文清洁并提供专业化指令。本页涵盖同步子 Agent（监控 Agent 等待子 Agent 完成）。

## 1. 环境准备与导入

In [ ]:
# pip install deepagents langchain langgraph tavily

from deepagents import create_deep_agent, CompiledSubAgent
from langchain.tools import tool
from typing import Literal
import os

## 2. 为什么使用子 Agent

子 Agent 解决**上下文膨胀问题**。当 Agent 使用输出较大的工具（网络搜索、文件读取、数据库查询）时，上下文窗口很快被中间结果填满。子 Agent 隔离这些细节工作——主 Agent 只接收最终结果，而非产生它的数十次工具调用。


**何时使用子 Agent：**
- ✅ 会混乱主 Agent 上下文的多步骤任务
- ✅ 需要自定义指令或工具的专业领域
- ✅ 需要不同模型能力的任务
- ✅ 希望主 Agent 专注于高层协调

**何时不使用子 Agent：**
- ❌ 简单的单步任务
- ❌ 需要维护中间上下文时
- ❌ 当开销超过收益时

## 3. 配置子 Agent

`subagents` 应为字典列表或 `CompiledSubAgent` 对象。

### 默认子 Agent

Deep Agents 自动添加一个同步通用子 Agent，除非你已提供一个同名的同步子 Agent。通用子 Agent 默认拥有文件系统工具，可通过额外工具/中间件自定义。

### 不使用子 Agent 运行

要在不使用 task 工具的情况下运行 Agent，需做两件事：
1. 在活动 harness profile 上设置 `general_purpose_subagent=GeneralPurposeSubagentProfile(enabled=False)`
2. 不通过 `subagents=` 向 `create_deep_agent` 传递同步子 Agent

## 4. 自定义子 Agent（字典方式）

| 字段 | 类型 | 描述 |
|------|------|------|
| `name` | str | 必填。唯一标识符。主 Agent 调用 `task()` 工具时使用此名称。 |
| `description` | str | 必填。子 Agent 的功能描述。要具体且以行动为导向。 |
| `system_prompt` | str | 必填。子 Agent 的指令。包含工具使用指导和输出格式要求。不从主 Agent 继承。 |
| `tools` | list[Callable] | 可选。子 Agent 可用的工具。保持最小化。默认从主 Agent 继承。指定时覆盖继承的工具。 |
| `model` | str \| BaseChatModel | 可选。覆盖主 Agent 的模型。省略时使用主 Agent 模型。 |
| `middleware` | list[Middleware] | 可选。用于自定义行为、日志或速率限制的额外中间件。不从主 Agent 继承。 |
| `interrupt_on` | dict | 可选。为特定工具配置人工审批。需要 checkpointer。默认从主 Agent 继承。 |
| `skills` | list[str] | 可选。技能源路径。子 Agent 拥有独立的 Skill 状态。 |
| `response_format` | ResponseFormat | 可选。子 Agent 的结构化输出模式。 |
| `permissions` | list[FilesystemPermission] | 可选。文件系统权限规则。默认从主 Agent 继承。 |

In [ ]:
# 定义搜索工具
@tool
def internet_search(query: str, max_results: int = 5) -> str:
    """执行网络搜索，返回搜索结果。"""
    # 此处应使用实际搜索 API，如 Tavily
    return f"关于 '{query}' 的搜索结果（共 {max_results} 条）..."


# 使用字典方式定义自定义子 Agent
research_subagent = {
    "name": "research-agent",
    "description": "用于深入调研问题",
    "system_prompt": "你是一个优秀的研究员，善于深入挖掘信息。",
    "tools": [internet_search],
    "model": "google_genai:gemini-3.5-flash",  # 可选覆盖，默认使用主 Agent 模型
}

subagents = [research_subagent]

# 创建深度 Agent
agent = create_deep_agent(
    model="google_genai:gemini-3.5-flash",
    subagents=subagents,
)

print("子 Agent 配置完成！")

## 5. 使用 CompiledSubAgent（编译子 Agent）

对于复杂工作流，可以使用预构建的 LangGraph 图作为 `CompiledSubAgent`。

In [ ]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

# 创建自定义 Agent 图
custom_graph = create_agent(
    model=ChatOpenAI(model="gpt-4"),
    tools=[internet_search],
    prompt="你是一个专门进行数据分析的 Agent...",
)

# 使用 CompiledSubAgent 包装
custom_subagent = CompiledSubAgent(
    name="data-analyzer",
    description="专门用于复杂数据分析任务的 Agent",
    runnable=custom_graph,
)

agent = create_deep_agent(
    model="google_genai:gemini-3.5-flash",
    tools=[internet_search],
    system_prompt="你是一个数据分析协调员",
    subagents=[custom_subagent],
)

print("CompiledSubAgent 配置完成！")

## 6. 结构化输出

子 Agent 支持结构化输出，父 Agent 接收可预测、可解析的 JSON 而非自由文本。需要 `deepagents>=0.5.3`。在子 Agent 配置上传递 `response_format`。

In [ ]:
from pydantic import BaseModel, Field

# 定义结构化输出模式
class ResearchFindings(BaseModel):
    """研究任务的结构化发现。"""
    summary: str = Field(description="发现摘要")
    confidence: float = Field(description="置信度分数（0 到 1）")
    sources: list[str] = Field(description="来源 URL 列表")

# 配置带结构化输出的子 Agent
research_subagent = {
    "name": "researcher",
    "description": "研究主题并返回结构化发现",
    "system_prompt": "深入研究给定主题，返回你的发现。",
    "tools": [internet_search],
    "response_format": ResearchFindings,  # 结构化输出
}

agent = create_deep_agent(
    model="google_genai:gemini-3.5-flash",
    subagents=[research_subagent],
)

print("结构化输出子 Agent 配置完成！")

## 7. 覆盖通用子 Agent

包含 `name="general-purpose"` 的子 Agent 可替换默认通用子 Agent。

In [ ]:
# 主 Agent 使用 Gemini；通用子 Agent 使用 GPT
agent = create_deep_agent(
    model="google_genai:gemini-3.5-flash",
    tools=[internet_search],
    subagents=[
        {
            "name": "general-purpose",  # 覆盖默认通用子 Agent
            "description": "用于研究和多步骤任务的通用 Agent",
            "system_prompt": "你是一个通用助手。",
            "tools": [internet_search],
            "model": "openai:gpt-4",  # 为委托任务使用不同模型
        },
    ],
)

print("通用子 Agent 已覆盖！")

## 8. 技能继承

- **通用子 Agent**：自动从主 Agent 继承技能
- **自定义子 Agent**：默认不继承技能——使用 `skills` 参数赋予技能

只有配置了 `skills` 的子 Agent 才会获得 `SkillsMiddleware` 实例。技能状态在父和子之间完全隔离。

In [ ]:
# 带独立技能的子 Agent
research_subagent = {
    "name": "researcher",
    "description": "具有专业技能的研究助手",
    "system_prompt": "你是一个研究员。",
    "tools": [internet_search],
    "skills": ["/skills/research/", "/skills/web-search/"],  # 子 Agent 专属技能
}

agent = create_deep_agent(
    model="google_genai:gemini-3.5-flash",
    skills=["/skills/main/"],  # 主 Agent 和通用子 Agent 获得这些
    subagents=[research_subagent],  # 只获得 /skills/research/ 和 /skills/web-search/
)

print("技能配置完成！")

## 9. 最佳实践

### 编写清晰描述

✅ 好的做法："分析财务数据并生成带置信度分数的投资见解"
❌ 不好的做法："处理财务事务"

### 保持系统提示详细

包含工具使用和输出格式的具体指导：

```python
research_subagent = {
    "system_prompt": """你是一个细致的研究员。你的工作是：
    1. 将研究问题分解为可搜索的查询
    2. 使用 internet_search 查找相关信息
    3. 将发现综合为全面但简洁的摘要
    4. 提出主张时引用来源

    输出格式：
    - 摘要（2-3 段）
    - 主要发现（要点）
    - 来源（带 URL）

    保持回答在 500 字以下以维护清洁上下文。""",
}
```

### 最小化工具集

只给子 Agent 所需的工具，提升专注度和安全性。

### 按任务选择模型

不同模型擅长不同任务：合同审查用大上下文模型，财务分析用数值分析强的模型。

### 返回简洁结果

指示子 Agent 返回摘要而非原始数据。

## 10. 多个专业子 Agent

为不同领域创建专业子 Agent。

In [ ]:
# 创建多个专业子 Agent
subagents = [
    {
        "name": "data-collector",
        "description": "从各种来源收集原始数据",
        "system_prompt": "全面收集关于主题的数据",
        "tools": [internet_search],
    },
    {
        "name": "data-analyzer",
        "description": "分析收集的数据以提取洞察",
        "system_prompt": "分析数据并提取关键洞察",
    },
    {
        "name": "report-writer",
        "description": "从分析结果撰写专业报告",
        "system_prompt": "从洞察创建专业报告",
    },
]

agent = create_deep_agent(
    model="google_genai:gemini-3.5-flash",
    system_prompt="你协调数据分析和报告流程。使用子 Agent 处理专业任务。",
    subagents=subagents,
)

print("多个专业子 Agent 配置完成！")

## 11. 上下文管理

运行上下文自动传播到所有子 Agent。每个子 Agent 运行接收父 invoke/ainvoke 调用时传递的相同运行上下文。

In [ ]:
from dataclasses import dataclass
from langchain.messages import HumanMessage
from langchain.tools import tool, ToolRuntime

# 定义上下文类型
@dataclass
class Context:
    user_id: str
    session_id: str

# 可访问上下文的工具
@tool
def get_user_data(query: str, runtime: ToolRuntime[Context]) -> str:
    """获取当前用户的数据。"""
    user_id = runtime.context.user_id
    return f"用户 {user_id} 的数据: {query}"

# 子 Agent 自动接收父 Agent 的上下文
research_subagent = {
    "name": "researcher",
    "description": "为当前用户进行研究",
    "system_prompt": "你是一个研究助手。",
    "tools": [get_user_data],
}

agent = create_deep_agent(
    model="google_genai:gemini-3.5-flash",
    subagents=[research_subagent],
    context_schema=Context,
)

print("上下文管理配置完成！")

---

**参考链接**
- [Deep Agents — Subagents](https://docs.langchain.com/oss/python/deepagents/subagents)
- [异步子 Agent](https://docs.langchain.com/oss/python/deepagents/async-subagents)
- [流式输出](https://docs.langchain.com/oss/python/deepagents/streaming)
- [结构化输出](https://docs.langchain.com/oss/python/deepagents/structured-output)
- [可观测性快速入门](https://docs.langchain.com/oss/python/deepagents/observability)